# 11 — Validate the susceptibility reporting mart

## Scope

The conventional susceptibility summary now uses the **earliest eligible culture per patient, cohort, and organism reporting group**, regardless of resistance phenotype. `overall` is the full first-culture population; MRSA/MSSA, VRE, and ESCE-R are subsets of that same population, not separate first-isolate selections.

The mart also applies cohort-specific organism–antibiotic approvals, reporting exclusions, and organism-level Gram classifications. Here I check the reporting grain, denominator arithmetic, subgroup counts, and whether the published pairs agree with the reference tables.

**Run order:** Notebook 10 validates `int_overall_first_reporting_isolate` and `int_overall_first_reporting_susceptibility`. Run this notebook after building the updated `mart_susceptibility` model and its dbt tests. Queries use the Databricks-connected `%sql` notebook environment.

The analysis is based on ARMD's categorical AST results; the project's 30-result display threshold and reporting suppressions are documented analytical choices, not a claim of CLSI M39 compliance.

## 1. Confirm the reporting version

The previous mart used a `base` category that excluded phenotype-specific records. The revised mart must contain `overall` rows and no legacy `base` rows.

In [1]:
%sql
select
    count(*) as mart_rows,
    sum(case when reporting_phenotype = 'overall' then 1 else 0 end)
        as overall_rows,
    sum(case when reporting_phenotype = 'base' then 1 else 0 end)
        as legacy_base_rows,
    case
        when sum(case when reporting_phenotype = 'overall'
                      then 1 else 0 end) > 0
         and sum(case when reporting_phenotype = 'base'
                      then 1 else 0 end) = 0
            then 'ready'
        else 'mart_not_migrated'
    end as mart_version_check
from workspace.micro_dev.mart_susceptibility;

,mart_rows,overall_rows,legacy_base_rows,mart_version_check
0,455,305,0,ready


**Current result:** 455 mart rows, 305 overall rows, zero legacy `base` rows. The remaining 150 rows are phenotype-specific subsets.

## 2. Check the reporting-pair seed

Only `reviewed` pairs with `include_primary_dashboard = 1` can reach the mart. Pairs still under review remain in the seed with `review_status = 'proposed'` and are inactive.

In [2]:
%sql
select
    review_status,
    include_primary_dashboard,
    count(*) as pair_rows

from workspace.micro_dev.reportable_organism_antibiotic_pairs

group by
    review_status,
    include_primary_dashboard

order by review_status;

,review_status,include_primary_dashboard,pair_rows
0,proposed,0,11
1,reviewed,1,311


**Current seed:** 311 reviewed/active pairs and 11 proposed/inactive pairs. The inactive rows remain available for later review without appearing in the antibiogram.

## 3. Validate grain and count reconciliation

One mart row represents a source release, cohort, organism reporting group, reporting phenotype, and antibiotic. I check the surrogate key, missing Gram metadata, negative values, and whether every eligible record falls into exactly one endpoint category.

In the current build, the mart has **455 unique rows**: 305 `overall` rows and 150 subgroup rows. All the structural error counts below are zero.

In [3]:
%sql
select
    count(*) as mart_rows,
    count(distinct susceptibility_mart_key) as unique_keys,
    sum(case when gram_stain_group is null then 1 else 0 end)
        as missing_gram_group,
    sum(case when eligible_count < 0
              or categorized_count < 0
              or untested_count < 0
             then 1 else 0 end) as negative_count_rows,
    sum(case when eligible_count <>
                  categorized_count + inconclusive_count
                  + synergism_count + missing_result_count
                  + conflict_result_count + untested_count
             then 1 else 0 end) as denominator_mismatches,
    sum(case when reporting_phenotype = 'base'
             then 1 else 0 end) as legacy_base_rows
from workspace.micro_dev.mart_susceptibility;

,mart_rows,unique_keys,missing_gram_group,negative_count_rows,denominator_mismatches,legacy_base_rows
0,455,455,0,0,0,0


### Natural-key duplicates

Check for duplicate organism–phenotype–antibiotic combinations within a source release and cohort. Any returned row would require investigation.

In [4]:
%sql
select
    source_version,
    cohort_id,
    organism_reporting_group,
    organism_name,
    reporting_phenotype,
    antibiotic,
    count(*) as row_count

from workspace.micro_dev.mart_susceptibility

group by
    source_version,
    cohort_id,
    organism_reporting_group,
    organism_name,
    reporting_phenotype,
    antibiotic

having count(*) > 1;

,source_version,cohort_id,organism_reporting_group,organism_name,reporting_phenotype,antibiotic,row_count


### Tested and untested counts

`eligible_count = tested_endpoint_count + untested_count`. This should return no rows.

In [5]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    eligible_count,
    tested_endpoint_count,
    untested_count

from workspace.micro_dev.mart_susceptibility

where tested_endpoint_count + untested_count <> eligible_count;

,cohort_id,organism_name,reporting_phenotype,antibiotic,eligible_count,tested_endpoint_count,untested_count


### Breakdown of tested endpoints

Reconcile tested endpoints into S/R/I, inconclusive, synergism, missing and conflict; reconcile categorized endpoints into S + I + R. Both comparisons should return no rows.

In [6]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    tested_endpoint_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count,
    inconclusive_count,
    synergism_count,
    missing_result_count,
    conflict_result_count

from workspace.micro_dev.mart_susceptibility

where tested_endpoint_count <>
      categorized_count
    + inconclusive_count
    + synergism_count
    + missing_result_count
    + conflict_result_count

   or categorized_count <>
      susceptible_count
    + resistant_count
    + intermediate_count;

,cohort_id,organism_name,reporting_phenotype,antibiotic,tested_endpoint_count,categorized_count,susceptible_count,resistant_count,intermediate_count,inconclusive_count,synergism_count,missing_result_count,conflict_result_count


### Component-count bounds

No individual susceptibility category can exceed its denominator. This should return no rows.

In [7]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    eligible_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count

from workspace.micro_dev.mart_susceptibility

where categorized_count > eligible_count
   or susceptible_count > categorized_count
   or resistant_count > categorized_count
   or intermediate_count > categorized_count;

,cohort_id,organism_name,reporting_phenotype,antibiotic,eligible_count,categorized_count,susceptible_count,resistant_count,intermediate_count


## 4. Enforce the reporting-pair policy

The mart should contain only active, reviewed organism–antibiotic pairs in the correct specimen cohort. The exclusion seed takes precedence over the allow-list; suppressed results remain available upstream for analysis.

### Inactive or wrong-cohort pairs

Find mart rows without an active matching `(organism_reporting_group, cohort_id, antibiotic)` entry. Expected: no rows.

In [8]:
%sql
select
    m.cohort_id,
    m.organism_reporting_group,
    m.reporting_phenotype,
    m.antibiotic
from workspace.micro_dev.mart_susceptibility m
left join workspace.micro_dev.reportable_organism_antibiotic_pairs p
    on m.organism_reporting_group = p.organism_reporting_group
   and m.cohort_id = p.cohort_id
   and m.antibiotic = p.antibiotic
   and p.include_primary_dashboard = 1
   and p.review_status = 'reviewed'
where p.organism_reporting_group is null;

,cohort_id,organism_reporting_group,reporting_phenotype,antibiotic


### Pairs suppressed by the exclusion seed

Find any excluded organism–antibiotic combination that leaked into the mart. Expected: no rows.

In [9]:
%sql
select
    m.cohort_id,
    m.organism_reporting_group,
    m.reporting_phenotype,
    m.antibiotic,
    e.exclusion_reason
from workspace.micro_dev.mart_susceptibility m
inner join workspace.micro_dev.organism_antibiotic_exclusions e
    on m.organism_reporting_group = e.organism_reporting_group
   and m.antibiotic = e.antibiotic;

,cohort_id,organism_reporting_group,reporting_phenotype,antibiotic,exclusion_reason


## 5. Reconstruct the overall denominator

For every approved, nonsuppressed pair, I independently count rows in `int_overall_first_reporting_isolate` and compare them with the mart's `overall` denominator. The full outer join also catches missing or unexpected overall pairs.

The overall isolate selection is independent of phenotype, so the eligible population includes first cultures flagged CRE or MDR. Expected: no mismatches.

In [10]:
%sql
with approved_pairs as (
    select
        p.organism_reporting_group,
        p.cohort_id,
        p.antibiotic
    from workspace.micro_dev.reportable_organism_antibiotic_pairs p
    left join workspace.micro_dev.organism_antibiotic_exclusions e
        on p.organism_reporting_group = e.organism_reporting_group
       and p.antibiotic = e.antibiotic
    where p.review_status = 'reviewed'
      and p.include_primary_dashboard = 1
      and e.organism_reporting_group is null
),
expected as (
    select
        f.source_version,
        f.cohort_id,
        f.organism_reporting_group,
        p.antibiotic,
        count(*) as expected_eligible
    from workspace.micro_dev.int_overall_first_reporting_isolate f
    inner join approved_pairs p
        on f.cohort_id = p.cohort_id
       and f.organism_reporting_group = p.organism_reporting_group
    group by
        f.source_version,
        f.cohort_id,
        f.organism_reporting_group,
        p.antibiotic
),
actual as (
    select
        source_version,
        cohort_id,
        organism_reporting_group,
        antibiotic,
        eligible_count
    from workspace.micro_dev.mart_susceptibility
    where reporting_phenotype = 'overall'
)
select
    coalesce(e.source_version, a.source_version) as source_version,
    coalesce(e.cohort_id, a.cohort_id) as cohort_id,
    coalesce(e.organism_reporting_group, a.organism_reporting_group)
        as organism_reporting_group,
    coalesce(e.antibiotic, a.antibiotic) as antibiotic,
    e.expected_eligible,
    a.eligible_count as actual_eligible
from expected e
full outer join actual a
    on e.source_version = a.source_version
   and e.cohort_id = a.cohort_id
   and e.organism_reporting_group = a.organism_reporting_group
   and e.antibiotic = a.antibiotic
where e.expected_eligible is null
   or a.eligible_count is null
   or e.expected_eligible <> a.eligible_count;

,source_version,cohort_id,organism_reporting_group,antibiotic,expected_eligible,actual_eligible


## 6. Check resistance-subgroup denominators

A subgroup is selected **from** the overall first-culture population. Its denominator cannot exceed the corresponding overall denominator, and an overall row must exist for every subgroup row.

In [11]:
%sql
select
    s.source_version,
    s.cohort_id,
    s.organism_reporting_group,
    s.reporting_phenotype,
    s.antibiotic,
    s.eligible_count as subgroup_eligible,
    o.eligible_count as overall_eligible
from workspace.micro_dev.mart_susceptibility s
left join workspace.micro_dev.mart_susceptibility o
    on s.source_version = o.source_version
   and s.cohort_id = o.cohort_id
   and s.organism_reporting_group = o.organism_reporting_group
   and s.antibiotic = o.antibiotic
   and o.reporting_phenotype = 'overall'
where s.reporting_phenotype <> 'overall'
  and (o.susceptibility_mart_key is null
       or s.eligible_count > o.eligible_count);

,source_version,cohort_id,organism_reporting_group,reporting_phenotype,antibiotic,subgroup_eligible,overall_eligible


### MRSA and MSSA subsets

For *S. aureus*, the oxacillin-resistant and oxacillin-susceptible groups must not overlap. Their combined denominator cannot exceed the overall *S. aureus* denominator. Records with unresolved oxacillin status are not assigned to MSSA.

In [12]:
%sql
with counts as (
    select
        source_version,
        cohort_id,
        organism_reporting_group,
        antibiotic,
        max(case when reporting_phenotype = 'overall'
                 then eligible_count end) as overall_n,
        sum(case when reporting_phenotype in (
                     'oxacillin_resistant', 'oxacillin_susceptible'
                 ) then eligible_count else 0 end) as mrsa_mssa_n
    from workspace.micro_dev.mart_susceptibility
    where organism_reporting_group = 'Staphylococcus aureus'
    group by
        source_version,
        cohort_id,
        organism_reporting_group,
        antibiotic
)
select *
from counts
where overall_n is null
   or mrsa_mssa_n > overall_n;

,source_version,cohort_id,organism_reporting_group,antibiotic,overall_n,mrsa_mssa_n


## 7. Validate display rules and Gram metadata

`display_status` is based on the number of categorized S/R/I endpoints. Pairs with fewer than 30 are retained for audit but marked `insufficient_data`; this is a project reporting threshold. Percent susceptible is calculated using categorized results, not all eligible isolates.

In [13]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    categorized_count,
    display_status

from workspace.micro_dev.mart_susceptibility

where (
        categorized_count >= 30
        and display_status <> 'display'
      )
   or (
        categorized_count < 30
        and display_status <> 'insufficient_data'
      );

,cohort_id,organism_name,reporting_phenotype,antibiotic,categorized_count,display_status


### Gram-group distribution

Gram classification comes from `organism_reporting_groups` and is attached using the base organism reporting group, so resistant subgroups inherit the same Gram classification.

This build contains 12 Gram-negative groups (308 mart rows) and 7 Gram-positive groups (147 mart rows).

In [14]:
%sql
select
    gram_stain_group,
    count(distinct organism_reporting_group) as organism_count,
    count(*) as mart_rows

from workspace.micro_dev.mart_susceptibility

group by gram_stain_group

order by gram_stain_group;

,gram_stain_group,organism_count,mart_rows
0,gram_negative,12,308
1,gram_positive,7,147


### Reference-data consistency

Check that every mart row has the expected Gram group according to the reporting-group seed. Expected: no rows.

In [15]:
%sql
select
    m.organism_reporting_group,
    m.organism_name,
    m.gram_stain_group as mart_group,
    g.gram_stain_group as reference_group
from workspace.micro_dev.mart_susceptibility m
left join workspace.micro_dev.organism_reporting_groups g
    on m.organism_reporting_group = g.organism_reporting_group
where m.gram_stain_group is null
   or g.organism_reporting_group is null
   or m.gram_stain_group <> g.gram_stain_group;

,organism_reporting_group,organism_name,mart_group,reference_group


## 8. Inspect the published result shape

### Blood-cohort preview

I review the blood results with both denominators and endpoint counts exposed. `display_status` should be used to filter the eventual Tableau antibiogram; the underlying counts remain available for QA.

In [16]:
%sql
select
    organism_reporting_group,
    organism_name,
    reporting_phenotype,
    antibiotic,
    eligible_count,
    tested_endpoint_count,
    untested_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count,
    inconclusive_count,
    synergism_count,
    missing_result_count,
    conflict_result_count,
    categorized_result_coverage_pct,
    susceptible_pct,
    resistant_pct,
    display_status

from workspace.micro_dev.mart_susceptibility

where cohort_id = 'blood'

order by
    organism_reporting_group,
    reporting_phenotype,
    pair_order;

,organism_reporting_group,organism_name,reporting_phenotype,antibiotic,eligible_count,tested_endpoint_count,untested_count,categorized_count,susceptible_count,resistant_count,intermediate_count,inconclusive_count,synergism_count,missing_result_count,conflict_result_count,categorized_result_coverage_pct,susceptible_pct,resistant_pct,display_status
0,Citrobacter freundii complex,Citrobacter freundii complex,overall,Ciprofloxacin,84,84,0,84,78,5,1,0,0,0,0,100.0,92.9,6.0,display
1,Citrobacter freundii complex,Citrobacter freundii complex,overall,Gentamicin,84,84,0,84,83,1,0,0,0,0,0,100.0,98.8,1.2,display
2,Citrobacter freundii complex,Citrobacter freundii complex,overall,Trimethoprim/Sulfamethoxazole,84,84,0,84,75,9,0,0,0,0,0,100.0,89.3,10.7,display
3,Citrobacter freundii complex,Citrobacter freundii complex,overall,Cefepime,84,50,34,50,49,1,0,0,0,0,0,59.5,98.0,2.0,display
4,Citrobacter freundii complex,Citrobacter freundii complex,overall,Meropenem,84,83,1,83,83,0,0,0,0,0,0,98.8,100.0,0.0,display
5,Citrobacter freundii complex,Citrobacter freundii complex,overall,Ertapenem,84,59,25,59,59,0,0,0,0,0,0,70.2,100.0,0.0,display
6,Citrobacter freundii complex,Citrobacter freundii complex,overall,Levofloxacin,84,81,3,81,76,3,2,0,0,0,0,96.4,93.8,3.7,display
7,Citrobacter koseri,Citrobacter koseri,overall,Ceftriaxone,44,44,0,44,44,0,0,0,0,0,0,100.0,100.0,0.0,display
8,Citrobacter koseri,Citrobacter koseri,overall,Ciprofloxacin,44,44,0,44,44,0,0,0,0,0,0,100.0,100.0,0.0,display
9,Citrobacter koseri,Citrobacter koseri,overall,Gentamicin,44,44,0,44,44,0,0,0,0,0,0,100.0,100.0,0.0,display


### Example: blood *E. coli* and ceftriaxone

This pair should have an `overall` row and a separate ESCE-R subset. The subgroup uses the overall population's selected first cultures rather than adding another isolate from the same patient.

In [17]:
%sql
select *

from workspace.micro_dev.mart_susceptibility

where cohort_id = 'blood'
  and organism_reporting_group = 'Escherichia coli'
  and antibiotic = 'Ceftriaxone'

order by reporting_phenotype;

,susceptibility_mart_key,source_version,cohort_id,organism_reporting_group,organism_name,reporting_phenotype,antibiotic,pair_order,eligible_count,tested_endpoint_count,categorized_count,susceptible_count,resistant_count,intermediate_count,inconclusive_count,synergism_count,missing_result_count,conflict_result_count,untested_count,categorized_result_coverage_pct,susceptible_pct,resistant_pct,display_status,gram_stain_group
0,cf1f8e76a9e509d27dded81f4f71674b,2025-04-11,blood,Escherichia coli,Escherichia coli — extended-spectrum cephalosporin resistant,esce_r,Ceftriaxone,1,467,466,466,1,461,4,0,0,0,0,1,99.8,0.2,98.9,display,gram_negative
1,8541e895d87b8fb6bd0358a78681727b,2025-04-11,blood,Escherichia coli,Escherichia coli,overall,Ceftriaxone,1,3022,3021,3016,2542,465,9,0,0,1,4,1,99.8,84.3,15.4,display,gram_negative


**Spot check:** blood *E. coli* has 3,022 overall eligible first cultures for ceftriaxone and 3,016 categorized endpoints (84.3% susceptible). The ESCE-R subset has 467 eligible first cultures, with 466 categorized endpoints (0.2% susceptible). The subset does not replace the overall row.

## 9. Reconcile approved pairs with the final mart

There are **311 active approved** organism–antibiotic–cohort pairs in the seed but **305 `overall` rows** in the mart. I check every approved pair missing from the `overall` output and classify the reason: an explicit suppression, no eligible first cultures, or an unexplained omission.

In this release, the six absent combinations are ceftriaxone and piperacillin/tazobactam for *Enterobacter cloacae* complex in each of the three cohorts. All six have `ampc_reporting_suppression`; none is unexplained.

In [18]:
%sql
with approved as (
    select
        organism_reporting_group,
        cohort_id,
        antibiotic
    from workspace.micro_dev.reportable_organism_antibiotic_pairs
    where review_status = 'reviewed'
      and include_primary_dashboard = 1
),
eligible as (
    select
        organism_reporting_group,
        cohort_id,
        count(*) as eligible_first_cultures
    from workspace.micro_dev.int_overall_first_reporting_isolate
    group by organism_reporting_group, cohort_id
)
select
    a.organism_reporting_group,
    a.cohort_id,
    a.antibiotic,
    e.exclusion_reason,
    coalesce(f.eligible_first_cultures, 0)
        as eligible_first_cultures,
    case
        when e.organism_reporting_group is not null
            then 'suppressed_by_policy'
        when coalesce(f.eligible_first_cultures, 0) = 0
            then 'no_eligible_first_cultures'
        else 'investigate'
    end as absence_reason
from approved a
left join workspace.micro_dev.organism_antibiotic_exclusions e
    on a.organism_reporting_group = e.organism_reporting_group
   and a.antibiotic = e.antibiotic
left join eligible f
    on a.organism_reporting_group = f.organism_reporting_group
   and a.cohort_id = f.cohort_id
left join workspace.micro_dev.mart_susceptibility m
    on a.organism_reporting_group = m.organism_reporting_group
   and a.cohort_id = m.cohort_id
   and a.antibiotic = m.antibiotic
   and m.reporting_phenotype = 'overall'
where m.susceptibility_mart_key is null
order by a.organism_reporting_group, a.cohort_id, a.antibiotic;

,organism_reporting_group,cohort_id,antibiotic,exclusion_reason,eligible_first_cultures,absence_reason
0,Enterobacter cloacae complex,blood,Ceftriaxone,ampc_reporting_suppression,305,suppressed_by_policy
1,Enterobacter cloacae complex,blood,Piperacillin/Tazobactam,ampc_reporting_suppression,305,suppressed_by_policy
2,Enterobacter cloacae complex,respiratory,Ceftriaxone,ampc_reporting_suppression,231,suppressed_by_policy
3,Enterobacter cloacae complex,respiratory,Piperacillin/Tazobactam,ampc_reporting_suppression,231,suppressed_by_policy
4,Enterobacter cloacae complex,urine,Ceftriaxone,ampc_reporting_suppression,1122,suppressed_by_policy
5,Enterobacter cloacae complex,urine,Piperacillin/Tazobactam,ampc_reporting_suppression,1122,suppressed_by_policy


**Coverage reconciliation:** All six missing approved pairs are documented AmpC suppressions for *E. cloacae* complex (ceftriaxone and piperacillin/tazobactam in blood, respiratory, and urine). There are no `investigate` cases.

## 10. Validation outcome

The saved results reconcile: **455 unique mart rows**, including **305 overall rows** and **150 phenotype-subgroup rows**. The 311 active approved pair rows reconcile to the 305 overall rows plus six intentionally suppressed combinations. No missing or unexpected approved pairs were found.

All notebook checks for grain, AST category totals, overall first-culture denominators, subgroup bounds, cohort-specific pair approval, suppression rules, the 30-result display flag, and Gram mapping returned no discrepancies. The separate `mart_mdro` retains its existing patient-level prevalence method.

**Reporting scope:** These are project-defined cumulative susceptibility summaries from ARMD categorical interpretations. The results should not be described as a formally CLSI-compliant clinical antibiogram without an additional standards and breakpoint review.